# Clinical Note Triage — from TF-IDF to transformer embeddings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/ms2a-machine-learning-practice/challenges/mlp-s7-clinical-note-triage.ipynb)

Read a free-text **medical transcription** and predict which of **12 clinical specialties**
(`spec_01` … `spec_12`) it belongs to: [challenge 174](https://ml-arena.com/viewchallenge/174)
on ML-Arena. Metric: **F1-macro**.

1. A **TF-IDF + logistic regression** baseline, submitted.
2. The same notes through a pretrained model from **Hugging Face**: text → tokens → embeddings
   → one vector per note.
3. Explore that vector space, then train an **MLP** on it.
4. Your turn: pick other models on Hugging Face, compare them, submit the best.

> In Colab: **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.
> The only thing you must edit is your API token (your **Profile** page on ml-arena.com).

## 0. Setup

In [ ]:
!pip install -q "mlarena-sdk>=3" transformers scikit-learn pandas matplotlib

import mlarena
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

API_TOKEN = "mlk_user_REPLACE_ME"   # <-- paste your token (Profile page)
CHALLENGE_ID = 174

client = mlarena.connect(api_key=API_TOKEN, base_url="https://ml-arena.com")

## 1. Get the data

`train.csv` holds `id, transcription, label`; `test.csv` holds `id, transcription`. Every model
below is scored on the **same** held-out 20 % of the training notes, so the numbers compare.

In [ ]:
client.download_dataset(CHALLENGE_ID, "data/")
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

y = train["label"].values
classes = sorted(set(y))
idx_tr, idx_val = train_test_split(np.arange(len(train)), test_size=0.2,
                                   random_state=0, stratify=y)
print("train:", train.shape, "  test:", test.shape)
print(train["label"].value_counts().sort_index().to_dict())

## 2. Baseline: TF-IDF + logistic regression

Each note becomes a sparse vector of word and bigram counts, down-weighted by how common
each term is. No notion of meaning: *myocardial* and *cardiac* are unrelated columns.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

text = train["transcription"]
baseline = make_pipeline(
    TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2), min_df=3,
                    max_features=40000, stop_words="english"),
    LogisticRegression(max_iter=2000, C=5.0, class_weight="balanced"))
baseline.fit(text.iloc[idx_tr], y[idx_tr])
print("validation F1-macro: %.3f" % f1_score(y[idx_val], baseline.predict(text.iloc[idx_val]), average="macro"))

Refit on all the training notes, predict the test set, submit.

In [ ]:
baseline.fit(text, y)
pd.DataFrame({"id": test["id"], "label": baseline.predict(test["transcription"])}).to_csv("submission.csv", index=False)
client.submit(challenge_id=CHALLENGE_ID, files=["submission.csv"], submission_name="tfidf-logreg")

## 3. Text → tokens

A transformer never reads words. Its **tokenizer** cuts the text into **subword tokens** from a
fixed vocabulary and maps each one to an integer **id**. We start with **GPT-2** (124 M
parameters, byte-level BPE, pretrained on general web text).

`MODEL_NAME` and `POOLING` are the two lines you will change in Section 7.

In [ ]:
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "openai-community/gpt2"
POOLING = "mean"                      # "mean" or "cls", see Section 4

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
sentence = "Laparoscopic cholecystectomy under general anesthesia, 40 mg IV."
print(tokenizer.tokenize(sentence))
print(tokenizer(sentence)["input_ids"])

n_tokens = train["transcription"].map(lambda t: len(tokenizer(t)["input_ids"]))
print("vocabulary:", len(tokenizer), "  tokens per note:", n_tokens.describe()[["50%", "max"]].to_dict())

**Questions**
- How is each medical word split? What does the `Ġ` prefix encode?
- What fraction of the notes is longer than 512 tokens? Below we **truncate** at 512: which
  part of a note is lost, and does it matter for guessing the specialty?

## 4. Tokens → one vector per note

The model looks up a vector for each token id, then mixes them through its attention layers.
`last_hidden_state` has shape `(batch, n_tokens, hidden_size)`: one **contextual** vector per
token. The classifier needs **one** vector per note, so we **pool**:

- **`cls`**: BERT-style encoders put a `[CLS]` token first and pretrain it to summarise the
  sequence, so take `h[:, 0]`.
- **`mean`**: average the token vectors, skipping padding with the attention mask $m_t$:

$$e = \frac{\sum_t m_t\, h_t}{\sum_t m_t}$$

GPT-2 is a decoder: each token attends only to the tokens before it, so its first token has
seen nothing of the note, and mean pooling is the natural choice.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()
if tokenizer.pad_token is None:          # GPT-2 was pretrained without padding
    tokenizer.pad_token = tokenizer.eos_token

@torch.no_grad()
def embed(texts, batch_size=16, max_length=512):
    out = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(texts[i:i + batch_size], padding=True, truncation=True,
                        max_length=max_length, return_tensors="pt").to(device)
        h = model(**enc).last_hidden_state                 # (batch, n_tokens, hidden)
        if POOLING == "cls":
            e = h[:, 0]
        else:
            m = enc["attention_mask"].unsqueeze(-1)         # 1 = token, 0 = padding
            e = (h * m).sum(1) / m.sum(1)
        out.append(e.float().cpu().numpy())
    return np.vstack(out)

X = embed(train["transcription"].tolist())
X_test = embed(test["transcription"].tolist())
print("embeddings:", X.shape, X_test.shape)

## 5. Explore the embedding space

Two notes are compared by the angle between their vectors, the **cosine similarity**
$\cos(a, b) = \frac{a \cdot b}{\lVert a \rVert\, \lVert b \rVert}$. Average the training
vectors of each specialty into a **centroid**, then classify a validation note by its most
similar centroid: no training at all.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

Z = X - X[idx_tr].mean(0)                # centre the cloud (see the questions)
centroids = np.stack([Z[idx_tr][y[idx_tr] == c].mean(0) for c in classes])
nearest = np.array(classes)[cosine_similarity(Z[idx_val], centroids).argmax(1)]
print("nearest-centroid validation F1-macro: %.3f" % f1_score(y[idx_val], nearest, average="macro"))

plt.imshow(cosine_similarity(centroids), cmap="RdBu_r", vmin=-1, vmax=1)
plt.xticks(range(12), classes, rotation=90); plt.yticks(range(12), classes)
plt.colorbar(); plt.title("Cosine similarity between specialty centroids"); plt.show()

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2).fit(Z[idx_tr])
P = pca.transform(Z[idx_tr])
for c in classes:
    plt.scatter(*P[y[idx_tr] == c].T, s=6, label=c)
plt.legend(markerscale=3, fontsize=7, bbox_to_anchor=(1, 1))
plt.title("PCA of the note embeddings (%.0f %% of the variance)" % (100 * pca.explained_variance_ratio_.sum()))
plt.show()

**Questions**
- Print `cosine_similarity(X[:100]).mean()` on the raw, uncentred embeddings. What do you
  get, and why does it make raw cosine similarity useless here?
- Which specialties are closest to each other? Are they also the ones the baseline confuses?
- Do the specialties form clusters in the PCA plot? How much of the variance do these two
  axes keep?

## 6. An MLP on the embeddings

The pretrained model stays frozen; only a small classifier is trained on its vectors.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

mlp = make_pipeline(StandardScaler(),
                    MLPClassifier(hidden_layer_sizes=(256,), alpha=1e-2, early_stopping=True,
                                  max_iter=500, random_state=0))
mlp.fit(X[idx_tr], y[idx_tr])
print("validation F1-macro: %.3f" % f1_score(y[idx_val], mlp.predict(X[idx_val]), average="macro"))

Submit it once it beats your previous best on the validation split.

In [ ]:
mlp.fit(X, y)
pd.DataFrame({"id": test["id"], "label": mlp.predict(X_test)}).to_csv("submission.csv", index=False)
client.submit(challenge_id=CHALLENGE_ID, files=["submission.csv"],
              submission_name=f"{MODEL_NAME.split('/')[-1]}-{POOLING}-mlp")

## 7. Your turn: choose the model

Change `MODEL_NAME` and `POOLING` in Section 3, rerun Sections 3 to 6, and keep a table:

| model | encoder / decoder | parameters | pretraining corpus | vocabulary | median tokens per note | pooling | validation F1 |
|---|---|---|---|---|---|---|---|

Some starting points (read each **model card** on huggingface.co before running it):

- general English: `google-bert/bert-base-uncased`, `distilbert/distilbert-base-uncased`,
  `FacebookAI/roberta-base`, `openai-community/gpt2-medium`, `Qwen/Qwen2.5-0.5B`
- trained for sentence similarity: `sentence-transformers/all-MiniLM-L6-v2`, `BAAI/bge-small-en-v1.5`
- scientific and biomedical text: `allenai/scibert_scivocab_uncased`,
  `dmis-lab/biobert-base-cased-v1.2`,
  `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext`,
  `NeuML/pubmedbert-base-embeddings`
- clinical notes: `emilyalsentzer/Bio_ClinicalBERT`

Or search the hub yourself: [feature-extraction](https://huggingface.co/models?pipeline_tag=feature-extraction)
and [sentence-similarity](https://huggingface.co/models?pipeline_tag=sentence-similarity) models.

**Questions**
- Which pooling suits each model, and why? Try both on one encoder: does the answer match?
- Tokenize the Section 3 sentence with each tokenizer. Does a vocabulary that keeps medical
  words whole go with a better score?
- Which matters more here, the size of the model or the corpus it was pretrained on?
- Which specialties does your best model still get wrong (`classification_report`)? Does
  TF-IDF get them right?
- The validation split has about 390 notes. How large must a gap in F1-macro be before you
  believe it?